In [1]:
# --- CONFIGURATION ---
# 1. Path to the root of your HRSC2016 dataset (e.g., the 'FullDataSet' folder)
HRSC_ROOT_PATH = '/caa/Homes01/mburges/datasets/hrsc2016_original' 

# 2. Path to the folder where you want to save all the images
OUTPUT_IMAGE_DIR = '/caa/Homes01/mburges/datasets/hrsc2016/images'

# 3. Path where you want to save the final COCO JSON file
OUTPUT_JSON_PATH = '/caa/Homes01/mburges/datasets/hrsc2016/annotations/hrsc2016_coco.json'

# --- SCRIPT ---

import os
import json
import shutil
import xml.etree.ElementTree as ET
from tqdm import tqdm

def create_coco_structure():
    """Initializes the basic COCO JSON structure."""
    return {
        "info": {
            "description": "HRSC2016 Dataset",
            "version": "1.0",
            "year": 2016,
            "contributor": "SKL-IEI, HUST",
            "date_created": "2016/10/20"
        },
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": []
    }

def convert_hrsc_to_coco():
    """
    Converts HRSC2016 dataset to COCO format.

    This script assumes the following directory structure for the HRSC2016 dataset:
    HRSC_ROOT_PATH/
    ├── AllImages/
    │   ├── 100000001.bmp
    │   └── ...
    └── Annotations/
        ├── 100000001.xml
        └── ...
    """
    
    # --- Setup ---
    # Define source paths based on the standard HRSC2016 structure
    source_image_dir = os.path.join(HRSC_ROOT_PATH, 'AllImages')
    source_annot_dir = os.path.join(HRSC_ROOT_PATH, 'Annotations')

    # Create output directories if they don't exist
    os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)
    os.makedirs(os.path.dirname(OUTPUT_JSON_PATH), exist_ok=True)

    # Initialize COCO data structure
    coco_data = create_coco_structure()
    
    # Initialize counters and mappings
    image_id_counter = 1
    annotation_id_counter = 1
    category_map = {}
    category_id_counter = 1
    
    # --- Main Loop ---
    xml_files = [f for f in os.listdir(source_annot_dir) if f.endswith('.xml')]
    print(f"Found {len(xml_files)} XML annotation files. Starting conversion...")

    for xml_file in tqdm(xml_files, desc="Converting HRSC to COCO"):
        try:
            tree = ET.parse(os.path.join(source_annot_dir, xml_file))
            root = tree.getroot()

            # --- Image Information ---
            img_filename_base = root.find('Img_FileName').text
            # The XML often specifies .bmp, but many versions use .png. We check for both.
            img_filename_bmp = f"{img_filename_base}.bmp"
            img_filename_png = f"{img_filename_base}.png"
            
            source_image_path_bmp = os.path.join(source_image_dir, img_filename_bmp)
            source_image_path_png = os.path.join(source_image_dir, img_filename_png)

            if os.path.exists(source_image_path_bmp):
                source_image_path = source_image_path_bmp
                final_img_filename = img_filename_png
            elif os.path.exists(source_image_path_png):
                source_image_path = source_image_path_png
                final_img_filename = img_filename_bmp
            else:
                print(f"Warning: Image for {xml_file} not found. Skipping.")
                continue

            width = int(root.find('Img_SizeWidth').text)
            height = int(root.find('Img_SizeHeight').text)

            # Copy image to the new directory
            shutil.copy2(source_image_path, os.path.join(OUTPUT_IMAGE_DIR, final_img_filename))
            
            # Add image entry to COCO data
            image_info = {
                "id": image_id_counter,
                "width": width,
                "height": height,
                "file_name": final_img_filename,
                "license": None,
                "flickr_url": "",
                "coco_url": "",
                "date_captured": "2000-01-01 00:00:00" # Placeholder
            }
            coco_data['images'].append(image_info)
            
            # --- Annotation Information ---
            objects = root.find('HRSC_Objects')
            if objects is None or len(objects) == 0:
                # No objects in this image, but we still keep the image entry
                image_id_counter += 1
                continue

            for obj in objects.findall('HRSC_Object'):
                # Category
                class_id_str = obj.find('Class_ID').text
                if class_id_str not in category_map:
                    category_map[class_id_str] = category_id_counter
                    category_info = {
                        "id": category_id_counter,
                        "name": class_id_str, # Using the ID as name, as HRSC does not provide text names
                        "supercategory": "ship" # All classes in HRSC2016 are ship-related
                    }
                    coco_data['categories'].append(category_info)
                    category_id_counter += 1
                
                category_id = category_map[class_id_str]

                # Bounding Box (axis-aligned)
                xmin = float(obj.find('box_xmin').text)
                ymin = float(obj.find('box_ymin').text)
                xmax = float(obj.find('box_xmax').text)
                ymax = float(obj.find('box_ymax').text)
                
                bbox_width = xmax - xmin
                bbox_height = ymax - ymin
                
                # COCO format is [xmin, ymin, width, height]
                coco_bbox = [xmin, ymin, bbox_width, bbox_height]
                area = bbox_width * bbox_height

                annotation_info = {
                    "id": annotation_id_counter,
                    "image_id": image_id_counter,
                    "category_id": category_id,
                    "bbox": coco_bbox,
                    "area": area,
                    "iscrowd": 0,
                    "segmentation": [] # HRSC provides rotated boxes, not segmentation masks
                }
                coco_data['annotations'].append(annotation_info)
                annotation_id_counter += 1
            
            image_id_counter += 1

        except ET.ParseError as e:
            print(f"Error parsing {xml_file}: {e}")
        except Exception as e:
            print(f"An unexpected error occurred with file {xml_file}: {e}")

    # --- Save JSON ---
    print(f"\nConversion complete. Saving COCO JSON file to {OUTPUT_JSON_PATH}")
    with open(OUTPUT_JSON_PATH, 'w') as f:
        json.dump(coco_data, f, indent=4)
    
    print("\n--- Summary ---")
    print(f"Total images processed: {len(coco_data['images'])}")
    print(f"Total annotations created: {len(coco_data['annotations'])}")
    print(f"Total categories found: {len(coco_data['categories'])}")
    print("-----------------")


if __name__ == '__main__':
    # Note: The original variable `FMoW_ROOT_PATH` was confusing as FMoW is a different dataset.
    # It has been renamed to `HRSC_ROOT_PATH` for clarity. Please ensure it points to the 
    # directory containing 'AllImages' and 'Annotations'.
    convert_hrsc_to_coco()

Found 1682 XML annotation files. Starting conversion...


Converting HRSC to COCO: 100%|██████████| 1682/1682 [00:02<00:00, 562.15it/s]


An unexpected error occurred with file sysdata.xml: 'NoneType' object has no attribute 'text'

Conversion complete. Saving COCO JSON file to /caa/Homes01/mburges/datasets/hrsc2016/annotations/hrsc2016_coco.json

--- Summary ---
Total images processed: 1680
Total annotations created: 2976
Total categories found: 28
-----------------
